# Aedex: get the model ready for the phone app

Our trained model `best.pt` runs in Python on a computer. A phone app can't run it. This notebook converts it to **TensorFlow Lite** (`.tflite`), a format made for phones, **checks that the conversion didn't break it**, and saves the files the app needs to Google Drive.

**Before you start**
1. Your Google Drive has the folder **`Dengue Breeding Site Scanner`** with `merged-v1.zip` and the training results in `runs/baseline-v1/weights/best.pt` (the training notebook put them there).
2. **No GPU needed.** Converting and checking runs fine on Colab's normal CPU, so you never wait for a GPU. Keep the runtime on **CPU**.
3. Run the cells **from top to bottom** (or menu **Runtime → Run all**).

**Time:** about 10 to 15 minutes.

| Step | What happens |
|---|---|
| 1 | Connect Google Drive |
| 2 | Settings |
| 3 | Install YOLO |
| 4 | Load the trained model |
| 5 | Convert to TensorFlow Lite (LiteRT) |
| 6 | Look inside the .tflite file |
| 7 | Check: same scores before and after? |
| 8 | Check with our own eyes, and make a scan record |
| 9 | Save the app files to Drive |

**When v2 (or any newer model) is ready:** change `MODEL_PT` and `VERSION` in Step 2 and run everything again. The app only needs the new files from Step 9.

## Step 1: Connect Google Drive

The trained model comes **from** Drive, and the phone files are saved **back to** Drive, where Member 3 picks them up.
Use the **same Google account** you trained with, and tick every permission box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Settings

| Setting | Meaning | Why this value |
|---|---|---|
| `MODEL_PT` | The trained model to convert | Model v1's best epoch |
| `VERSION` | Name tag for the output files | Files become `aedex-v1.tflite`, so v1 and v2 never get mixed up |
| `ZIP_NAME` | The dataset zip, only to borrow its **test** photos for the check | The check must use photos the model never trained on |
| `IMGSZ` | Image size the model expects | Must match training: 640 |
| `CONF` | Minimum confidence to show a box in the app | **0.15**, lower than YOLO's usual 0.25: it finds more containers with a few more false alarms. Recall matters most |
| `IOU` | How much two boxes may overlap before one is treated as a duplicate | 0.45 is YOLO's usual value for apps |

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/Dengue Breeding Site Scanner'
MODEL_PT  = f'{DRIVE_DIR}/runs/baseline-v1/weights/best.pt'
VERSION   = 'v1'
ZIP_NAME  = 'merged-v1.zip'

IMGSZ = 640
CONF  = 0.15
IOU   = 0.45

OUT_DIR = f'{DRIVE_DIR}/app-model'   # the folder you share with Member 3

import pathlib
for p in [MODEL_PT, f'{DRIVE_DIR}/{ZIP_NAME}']:
    print('OK     ' if pathlib.Path(p).exists() else 'MISSING', p)

## Step 3: Install YOLO

Same package as in training (`ultralytics`). During Step 5 it also installs the converter tools it needs by itself.

In [ ]:
%pip install -q ultralytics

## Step 4: Load the trained model

We copy `best.pt` to Colab's own disk first, so the conversion's temporary files are not written into your Drive.

The **class list** printed here is the most important fact for the app: the model never outputs the word "Tire", only the **number 3**. The app turns numbers back into names with this list, so it must be in exactly this order.

In [ ]:
import shutil
from ultralytics import YOLO

WORK = pathlib.Path('/content/export')
WORK.mkdir(exist_ok=True)
pt_path = WORK / f'aedex-{VERSION}.pt'
shutil.copy(MODEL_PT, pt_path)

model = YOLO(str(pt_path))
NAMES = [model.names[i] for i in range(len(model.names))]
for i, n in enumerate(NAMES):
    print(i, n)

## Step 5: Convert to TensorFlow Lite (LiteRT)

Google renamed TensorFlow Lite to **LiteRT** ("Lite Runtime"), so Ultralytics calls this format `litert`. The file is still a `.tflite`, and phone plugins read it the same way. The export prints a lot; that's normal. Takes about 3 to 8 minutes.

`QUANTIZE` decides how precisely the model's numbers are stored:
- `None`: full **32-bit** numbers (FP32). Nothing is lost; about 10 MB. **Our choice.**
- `'w8a32'`: weights squeezed to **8-bit**. About 4 times smaller and faster, but the scores can drop a little. Worth trying later if the app is slow; Step 7 would tell us how much it costs.

In [ ]:
QUANTIZE = None   # None = full 32-bit precision (safest). Try 'w8a32' later for a smaller file.

exported = model.export(format='litert', imgsz=IMGSZ, quantize=QUANTIZE)
print('Export returned:', exported)

exp = pathlib.Path(exported)
search_dir = exp if exp.is_dir() else exp.parent
tflite_files = sorted(search_dir.rglob('*.tflite'))
for f in tflite_files:
    print(f'{f.name:35s} {f.stat().st_size / 1e6:5.1f} MB')

TFLITE = next((f for f in tflite_files if 'float32' in f.name), tflite_files[0])
print('Using:', TFLITE)

## Step 6: Look inside the .tflite file

The app developer needs to know what goes **in** and what comes **out**:
- **Input:** one photo, resized to 640 × 640, 3 colour channels (RGB), values from 0 to 1.
- **Output:** for each of the **8,400** places the model looks at, 4 box numbers (centre x, centre y, width, height) plus one score per class. With 7 classes that is 4 + 7 = **11** numbers each. The app (or its YOLO plugin) keeps the high scores and removes duplicate boxes.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter   # LiteRT, installed by the export
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

interp = Interpreter(model_path=str(TFLITE))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
INPUT_SHAPE  = [int(x) for x in inp['shape']]
OUTPUT_SHAPE = [int(x) for x in out['shape']]
print('Input :', INPUT_SHAPE,  inp['dtype'].__name__)
print('Output:', OUTPUT_SHAPE, out['dtype'].__name__)
print('Expected output: [1,', 4 + len(NAMES), ', 8400]')

## Step 7: Check: same scores before and after?

A conversion can silently break a model (wrong colours, wrong scaling). So we score **both** versions on the same 235 **test** photos and compare.

We take only the `test/` folder out of the dataset zip, which is much faster than unzipping all 4,979 images.

**Pass:** precision, recall and mAP50 of the `.tflite` are within about **0.02** of the `.pt`. Recall should match the 0.735 from training.

In [ ]:
import zipfile, yaml

TEST_DIR = pathlib.Path('/content/testset')
with zipfile.ZipFile(f'{DRIVE_DIR}/{ZIP_NAME}') as z:
    members = [m for m in z.namelist() if m.startswith('test/')]
    z.extractall(TEST_DIR, members)
n_images = len(list((TEST_DIR / 'test' / 'images').glob('*')))
print('Test photos:', n_images)

# A small dataset file that points at the test photos only.
test_yaml = TEST_DIR / 'test.yaml'
test_yaml.write_text(yaml.safe_dump({
    'path': str(TEST_DIR),
    'train': 'test/images',
    'val': 'test/images',
    'names': {i: n for i, n in enumerate(NAMES)},
}))

def score(path):
    m = YOLO(str(path), task='detect')
    r = m.val(data=str(test_yaml), imgsz=IMGSZ, batch=1, device='cpu', plots=False, verbose=False)
    return {'precision': float(r.box.mp), 'recall': float(r.box.mr), 'mAP50': float(r.box.map50)}

scores_pt = score(pt_path)
scores_tflite = score(TFLITE)

print(f"\n{'':10s}{'.pt':>10s}{'.tflite':>10s}{'diff':>10s}")
ok = True
for k in ['precision', 'recall', 'mAP50']:
    d = scores_tflite[k] - scores_pt[k]
    ok = ok and abs(d) <= 0.02
    print(f'{k:10s}{scores_pt[k]:10.3f}{scores_tflite[k]:10.3f}{d:+10.3f}')
print('\nPASS: the phone model scores like the original.' if ok
      else '\nCHECK: the scores differ by more than 0.02. Send this output to the team before using the file.')

## Step 8: Check with our own eyes, and make a scan record

Scores can hide mistakes, so we also **look**: the `.tflite` model runs on 3 test photos with the app's settings (`CONF` 0.15).

Under each photo is the **scan record** the app will build from the boxes: just a count per class. This is the `counts` field from Member 3's scan data contract (`docs/tasks/member-3-hotspot-map.md`).

In [ ]:
import json, random
from collections import Counter
import matplotlib.pyplot as plt

phone_model = YOLO(str(TFLITE), task='detect')
photos = sorted((TEST_DIR / 'test' / 'images').glob('*'))
random.seed(7)
for photo in random.sample(photos, 3):
    r = phone_model.predict(str(photo), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
    counts = dict(Counter(NAMES[int(c)] for c in r.boxes.cls))
    plt.figure(figsize=(7, 7))
    plt.imshow(r.plot()[:, :, ::-1])   # YOLO draws in BGR; matplotlib wants RGB
    plt.axis('off')
    plt.title(photo.name[:60], fontsize=8)
    plt.show()
    print('counts:', json.dumps(counts))

## Step 9: Save the app files to Drive

Three files go into **`Dengue Breeding Site Scanner/app-model/`** in your Drive:

| File | What it is |
|---|---|
| `aedex-v1.tflite` | The model for the phone |
| `labels.txt` | The class names, one per line, **in number order**. The app reads names from here instead of hard-coding them, so v2's new class 7 (Pail-Basin) needs no code change |
| `aedex-v1-info.json` | Everything else the app needs: image size, thresholds, input/output shapes, and the check scores |

Then **share the `app-model` folder** with Member 3 (right-click → Share → Viewer).

In [ ]:
import datetime

OUT = pathlib.Path(OUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy(TFLITE, OUT / f'aedex-{VERSION}.tflite')
(OUT / 'labels.txt').write_text('\n'.join(NAMES) + '\n')
info = {
    'version': VERSION,
    'model_file': f'aedex-{VERSION}.tflite',
    'labels_file': 'labels.txt',
    'classes': NAMES,
    'imgsz': IMGSZ,
    'conf_threshold': CONF,
    'iou_threshold': IOU,
    'input_shape': INPUT_SHAPE,
    'output_shape': OUTPUT_SHAPE,
    'quantize': 'none (float32)' if QUANTIZE is None else str(QUANTIZE),
    'test_scores_pt': scores_pt,
    'test_scores_tflite': scores_tflite,
    'exported': datetime.date.today().isoformat(),
}
(OUT / f'aedex-{VERSION}-info.json').write_text(json.dumps(info, indent=2))

for f in sorted(OUT.iterdir()):
    print(f'{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB')
print('\nDone. Share the app-model folder with Member 3.')